# Activist Codes 需求分析与数据科学解决方案

这份文件按照 notebook 风格组织：
- 先还原业务需求
- 再看现有数据能支持什么
- 然后给出数据证据
- 最后提出可以执行的数据科学 / 数据治理方案

背景来自 2025-08-20 的邮件讨论，核心问题包括：
1. Activist Codes 和 Tags 到底应该怎么分工？
2. 上传名单时，应该如何统一打标签？
3. 当前已有的 Activist Codes / Tags 是否存在重复、命名不一致、需要合并清理的问题？
4. 能不能建立一套后续可持续治理的规范？


In [ ]:
from pathlib import Path
import sqlite3
from html import escape


def find_repo_root():
    if "__file__" in globals():
        start = Path(__file__).resolve().parent
    else:
        start = Path.cwd().resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "workbench" / "everyaction_workbench.sqlite").exists():
            return candidate

    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
DB_PATH = ROOT / "workbench" / "everyaction_workbench.sqlite"
FIGURE_DIR = ROOT / "analysis" / "figures"
REPORT_PATH = ROOT / "analysis" / "activist_codes_requirements_report.md"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def query_all(conn, sql, params=()):
    conn.row_factory = sqlite3.Row
    rows = conn.execute(sql, params).fetchall()
    return [dict(row) for row in rows]


def query_one(conn, sql, params=()):
    rows = query_all(conn, sql, params)
    return rows[0] if rows else {}


def markdown_table(rows, columns):
    if not rows:
        return "_No rows returned._"

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, separator, *body])


def write_svg_bar_chart(path, title, items, width=980, height=420, bar_color="#2f6fed"):
    if not items:
        return

    left = 80
    right = 40
    top = 70
    bottom = 90
    plot_width = width - left - right
    plot_height = height - top - bottom
    max_value = max(value for _label, value in items) or 1
    gap = 18
    bar_width = max(22, int((plot_width - gap * (len(items) - 1)) / max(1, len(items))))

    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<style>text { font-family: Arial, sans-serif; fill: #222; } .small { font-size: 12px; } .label { font-size: 12px; } .title { font-size: 20px; font-weight: bold; }</style>',
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#ffffff"/>',
        f'<text x="{width/2}" y="30" text-anchor="middle" class="title">{escape(title)}</text>',
        f'<line x1="{left}" y1="{top + plot_height}" x2="{left + plot_width}" y2="{top + plot_height}" stroke="#444" stroke-width="1"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_height}" stroke="#444" stroke-width="1"/>',
    ]

    for tick in range(5):
        value = max_value * tick / 4
        y = top + plot_height - (plot_height * tick / 4)
        parts.append(f'<line x1="{left - 4}" y1="{y}" x2="{left + plot_width}" y2="{y}" stroke="#e9e9e9" stroke-width="1"/>')
        parts.append(f'<text x="{left - 10}" y="{y + 4}" text-anchor="end" class="small">{int(round(value))}</text>')

    x = left
    for label, value in items:
        bar_height = 0 if max_value == 0 else plot_height * value / max_value
        y = top + plot_height - bar_height
        parts.append(f'<rect x="{x}" y="{y}" width="{bar_width}" height="{bar_height}" fill="{bar_color}" rx="4"/>')
        parts.append(f'<text x="{x + bar_width/2}" y="{y - 8}" text-anchor="middle" class="small">{value}</text>')
        parts.append(f'<text x="{x + bar_width/2}" y="{top + plot_height + 18}" text-anchor="middle" class="label">{escape(str(label))}</text>')
        x += bar_width + gap

    parts.append("</svg>")
    path.write_text("\n".join(parts), encoding="utf-8")


## 1. 把邮件内容拆成结构化需求

这一步很重要，因为邮件里的表述是业务语言，不是数据问题定义。
我们先把它翻译成可以落地的需求。


In [ ]:
requirements = [
    {
        "requirement_id": "R1",
        "business_need": "建立 Activist Codes 的命名规范",
        "why_it_matters": "上传更多名单之前，需要统一 label 规则，避免后续越积越乱。",
        "data_question": "现有 code 是否存在命名混乱、重复、不可扩展的问题？",
    },
    {
        "requirement_id": "R2",
        "business_need": "明确 Activist Codes 与 Tags 的分工",
        "why_it_matters": "Linda 明确怀疑 Tags 之前没有正确使用。",
        "data_question": "哪些标签应该是 record-level code，哪些应该是 roll-up tag？",
    },
    {
        "requirement_id": "R3",
        "business_need": "为上传名单建立多标签规则",
        "why_it_matters": "例如 Maricopa County PCs 上传时，是否应该打两个 Activist Codes？",
        "data_question": "现有 code 体系是否支持“角色 + 地理 + 年份/来源”的组合标注？",
    },
    {
        "requirement_id": "R4",
        "business_need": "清理重复和重叠",
        "why_it_matters": "邮件明确提到 Tags 有 duplicate，且 Tags 与 Activist Codes 之间可能重叠。",
        "data_question": "现有 Activist Codes 是否已经出现重复、短名冲突、低价值存量？",
    },
    {
        "requirement_id": "R5",
        "business_need": "建立后续可治理的体系",
        "why_it_matters": "如果不先定规则，后续每次上传名单都会制造新的治理成本。",
        "data_question": "能否建立一张 canonical dictionary 和自动质检流程？",
    },
]

print(markdown_table(requirements, ["requirement_id", "business_need", "why_it_matters", "data_question"]))


### 结果解读

从邮件可以提炼出 5 个真正的业务需求。
这不是一个单纯的“导入名单”问题，而是一个：
- 命名治理
- 标签体系设计
- 历史清理
- 未来自动化
的综合数据治理问题。


## 2. 看现有数据到底能支持哪些分析

当前工作区里，和这个议题直接相关的有三张表：
- `activist_codes_list_xlsx`：更新后的 Activist Code 定义表
- `all_activist_codes_applied`：Activist Code 应用表
- `tags_list`：Tag 定义表

这意味着我们现在不仅能分析 Activist Codes，
也能开始对 Tags 本身做重复、层级和重叠审计。


In [ ]:
conn = sqlite3.connect(DB_PATH)

data_availability = query_one(
    conn,
    """
    SELECT
        (SELECT COUNT(*) FROM activist_codes_list_xlsx) AS activist_code_definitions,
        (SELECT COUNT(*) FROM all_activist_codes_applied) AS activist_code_applications,
        (SELECT COUNT(DISTINCT activistcodeid) FROM all_activist_codes_applied) AS distinct_applied_code_ids,
        (SELECT COUNT(DISTINCT activistcodename) FROM all_activist_codes_applied) AS distinct_applied_code_names,
        (SELECT COUNT(*) FROM tags_list) AS tag_definitions,
        (SELECT COUNT(*) FROM (
            SELECT name FROM tags_list GROUP BY name HAVING COUNT(*) > 1
        )) AS duplicate_tag_name_groups,
        (SELECT COUNT(*) FROM tags_list t WHERE EXISTS (
            SELECT 1 FROM activist_codes_list_xlsx a WHERE a.long_name = t.name
        )) AS exact_tag_code_name_overlap
    """
)

data_availability


### 结果解读

这一步确认了两个事实：
1. Activist Codes 和 Tags 现在都已经有数据可分析
2. 团队担心的 Tag duplicate 和 Tag/Code overlap 现在都可以做实证审计


In [ ]:
print(f"Definitions in activist_codes_list_xlsx: {data_availability['activist_code_definitions']:,}")
print(f"Applications in all_activist_codes_applied: {data_availability['activist_code_applications']:,}")
print(f"Distinct applied code IDs: {data_availability['distinct_applied_code_ids']:,}")
print(f"Distinct applied code names: {data_availability['distinct_applied_code_names']:,}")
print(f"Definitions in tags_list: {data_availability['tag_definitions']:,}")
print(f"Duplicate tag-name groups: {data_availability['duplicate_tag_name_groups']:,}")
print(f"Exact tag/code name overlaps: {data_availability['exact_tag_code_name_overlap']:,}")
print()
print(
    "Interpretation: both Activist Code and Tag data are now available. "
    "That means this is no longer just a naming-framework exercise; it is now a real overlap and cleanup project."
)


## 3. 先看字典表和应用表的基本结构

我们先看：
- 定义表按 type 怎么分布
- 应用表按 type 怎么分布

这一步能帮助我们判断：
当前体系更偏“治理型目录”，还是更偏“实际运营行为”。


In [ ]:
definition_type_rows = query_all(
    conn,
    """
    SELECT type, COUNT(*) AS definition_count
    FROM activist_codes_list_xlsx
    GROUP BY type
    ORDER BY definition_count DESC
    """
)

application_type_rows = query_all(
    conn,
    """
    SELECT activistcodetype, COUNT(*) AS application_count
    FROM all_activist_codes_applied
    GROUP BY activistcodetype
    ORDER BY application_count DESC
    """
)

print(markdown_table(definition_type_rows, ["type", "definition_count"]))
print()
print(markdown_table(application_type_rows, ["activistcodetype", "application_count"]))


### 结果解读

如果某个 type 在定义表很多、应用表也很多，说明这是核心运营维度。
如果定义很多但应用很少，说明可能有“字典膨胀”或历史遗留。


In [ ]:
top_definition_type = definition_type_rows[0]
top_application_type = application_type_rows[0]

print(
    f"Most common definition type: {top_definition_type['type']} ({top_definition_type['definition_count']} codes)"
)
print(
    f"Most heavily applied type: {top_application_type['activistcodetype']} ({top_application_type['application_count']:,} applications)"
)
print()
print(
    "Interpretation: Constituency codes dominate both the catalog and real-world usage. "
    "That means list-building and audience grouping are already the main use case of this system."
)


## 4. 检查命名治理问题：short name 冲突

Linda 在邮件里已经提到：
- Long name 比较描述性
- Medium / short 因为字符限制，经常“看不懂但必须填”

这在数据治理里是一个危险信号，因为 short name 如果重复，就不应该被拿来做 canonical key。


In [ ]:
short_conflict_rows = query_all(
    conn,
    """
    SELECT short, COUNT(*) AS duplicate_count, GROUP_CONCAT(long_name, ' | ') AS long_names
    FROM activist_codes_list_xlsx
    GROUP BY short
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC, short
    LIMIT 15
    """
)

print(markdown_table(short_conflict_rows, ["short", "duplicate_count", "long_names"]))


### 结果解读

这个表用来判断：
short name 能不能作为对外规范、导出主键或者人工识别依据。


In [ ]:
short_conflict_summary = query_one(
    conn,
    """
    SELECT
        COUNT(*) AS duplicate_short_groups,
        SUM(duplicate_count - 1) AS extra_rows
    FROM (
        SELECT short, COUNT(*) AS duplicate_count
        FROM activist_codes_list_xlsx
        GROUP BY short
        HAVING COUNT(*) > 1
    )
    """
)

print(short_conflict_summary)
print()
print(
    "Interpretation: short names are clearly not unique, so they should never be treated as the system's canonical identifier. "
    "Long name or a formal code ID dictionary is the safer governance key."
)


## 5. 检查定义表与应用表能否对上

这是治理项目里非常关键的一步：
如果定义表和应用表对不上，后续所有 consolidation 都会很痛苦。

当前限制是：
`activist_codes_list_xlsx` 仍然没有 `activistcodeid`，
所以我们只能先用 `type + long_name` 去和应用表里的 `type + activistcodename` 做匹配。


In [ ]:
match_summary = query_one(
    conn,
    """
    WITH applied_codes AS (
      SELECT DISTINCT activistcodetype, activistcodename
      FROM all_activist_codes_applied
    )
    SELECT
      SUM(CASE WHEN EXISTS (
        SELECT 1 FROM activist_codes_list_xlsx l
        WHERE l.type = a.activistcodetype AND l.long_name = a.activistcodename
      ) THEN 1 ELSE 0 END) AS match_long_name,
      SUM(CASE WHEN EXISTS (
        SELECT 1 FROM activist_codes_list_xlsx l
        WHERE l.type = a.activistcodetype AND l.medium = a.activistcodename
      ) THEN 1 ELSE 0 END) AS match_medium,
      SUM(CASE WHEN EXISTS (
        SELECT 1 FROM activist_codes_list_xlsx l
        WHERE l.type = a.activistcodetype AND l.short = a.activistcodename
      ) THEN 1 ELSE 0 END) AS match_short,
      COUNT(*) AS distinct_applied_codes
    FROM applied_codes a
    """
)

print(match_summary)

unmatched_applied_rows = query_all(
    conn,
    """
    WITH applied_codes AS (
      SELECT DISTINCT activistcodetype, activistcodename
      FROM all_activist_codes_applied
    ), matched AS (
      SELECT
        a.activistcodetype,
        a.activistcodename,
        CASE WHEN EXISTS (
          SELECT 1 FROM activist_codes_list_xlsx l
          WHERE l.type = a.activistcodetype AND l.long_name = a.activistcodename
        ) THEN 1 ELSE 0 END AS long_match,
        CASE WHEN EXISTS (
          SELECT 1 FROM activist_codes_list_xlsx l
          WHERE l.type = a.activistcodetype AND l.medium = a.activistcodename
        ) THEN 1 ELSE 0 END AS medium_match,
        CASE WHEN EXISTS (
          SELECT 1 FROM activist_codes_list_xlsx l
          WHERE l.type = a.activistcodetype AND l.short = a.activistcodename
        ) THEN 1 ELSE 0 END AS short_match
      FROM applied_codes a
    )
    SELECT activistcodetype, activistcodename
    FROM matched
    WHERE long_match = 0 AND medium_match = 0 AND short_match = 0
    """
)

print(markdown_table(unmatched_applied_rows, ["activistcodetype", "activistcodename"]))


### 结果解读

这是一个好消息也是一个警告：
- 好消息：绝大多数应用中的 code 都能按 long name 对上定义表
- 警告：定义表缺少正式 ID，会让后续治理高度依赖名字匹配


In [ ]:
long_name_match_rate = round(match_summary["match_long_name"] / match_summary["distinct_applied_codes"] * 100, 2)
print(f"Long-name match rate between applied codes and definitions: {long_name_match_rate}%")
print()
print(
    "Interpretation: this governance project is feasible because name-based reconciliation is already strong. "
    "But the next version of the code dictionary should include an explicit ID key so future audits do not depend on string matching."
)


## 6. 看定义表里有哪些 code 根本没在应用表出现

这一步用来识别“存量僵尸 code”。
它们不一定都该删除，但至少需要标记：
- 是否废弃
- 是否待启用
- 是否只是历史遗留


In [ ]:
unused_definition_summary = query_one(
    conn,
    """
    WITH defs AS (
      SELECT type, long_name
      FROM activist_codes_list_xlsx
    ), applied AS (
      SELECT DISTINCT activistcodetype, activistcodename
      FROM all_activist_codes_applied
    )
    SELECT
      COUNT(*) AS total_defs,
      SUM(CASE WHEN EXISTS (
        SELECT 1 FROM applied a
        WHERE a.activistcodetype = d.type AND a.activistcodename = d.long_name
      ) THEN 1 ELSE 0 END) AS defs_seen_in_applied,
      SUM(CASE WHEN NOT EXISTS (
        SELECT 1 FROM applied a
        WHERE a.activistcodetype = d.type AND a.activistcodename = d.long_name
      ) THEN 1 ELSE 0 END) AS defs_not_seen_in_applied
    FROM defs d
    """
)

unused_definition_examples = query_all(
    conn,
    """
    WITH defs AS (
      SELECT type, long_name, medium, short
      FROM activist_codes_list_xlsx
    ), applied AS (
      SELECT DISTINCT activistcodetype, activistcodename
      FROM all_activist_codes_applied
    )
    SELECT d.type, d.long_name, d.medium, d.short
    FROM defs d
    WHERE NOT EXISTS (
      SELECT 1 FROM applied a
      WHERE a.activistcodetype = d.type AND a.activistcodename = d.long_name
    )
    ORDER BY d.type, d.long_name
    """
)

print(unused_definition_summary)
print()
print(markdown_table(unused_definition_examples, ["type", "long_name", "medium", "short"]))


### 结果解读

少量“未出现定义”本身不可怕，
可怕的是没有治理流程去决定它们应该：
- 保留
- 合并
- 废弃
- 重命名


In [ ]:
print(
    "Interpretation: the unused-code count is small enough to clean manually. "
    "This is encouraging, because it means the main governance work is not mass deletion; it is standardization and operating rules."
)


## 7. 直接审计 Tags：重复、层级和与 Activist Codes 的重叠

Linda 在邮件里点名提到了两个担心：
- Tag 里有 duplicate
- Tags 和 Activist Codes 之间可能有重复或重叠

现在 Tag 数据已经到了，所以这一步可以直接验证。


In [ ]:
tags_summary = query_one(
    conn,
    """
    SELECT
      COUNT(*) AS total_tags,
      SUM(CASE WHEN instr(path, '/') > 0 THEN 1 ELSE 0 END) AS hierarchical_paths,
      SUM(CASE WHEN path = name THEN 1 ELSE 0 END) AS path_equals_name,
      SUM(CASE WHEN COALESCE(description, '') = '' THEN 1 ELSE 0 END) AS missing_description,
      (SELECT COUNT(*) FROM (
          SELECT name FROM tags_list GROUP BY name HAVING COUNT(*) > 1
      )) AS duplicate_tag_name_groups,
      (SELECT COALESCE(SUM(cnt - 1), 0) FROM (
          SELECT COUNT(*) AS cnt FROM tags_list GROUP BY name HAVING COUNT(*) > 1
      )) AS duplicate_tag_extra_rows,
      (SELECT COUNT(*) FROM tags_list t WHERE EXISTS (
          SELECT 1 FROM activist_codes_list_xlsx a WHERE a.long_name = t.name
      )) AS exact_name_overlap_with_activist_codes
    FROM tags_list
    """
)

duplicate_tag_rows = query_all(
    conn,
    """
    SELECT name, COUNT(*) AS duplicate_count, GROUP_CONCAT(id, ' | ') AS ids
    FROM tags_list
    GROUP BY name
    HAVING COUNT(*) > 1
    """
)

overlap_tag_rows = query_all(
    conn,
    """
    SELECT t.name, t.path, t.description
    FROM tags_list t
    WHERE EXISTS (
      SELECT 1 FROM activist_codes_list_xlsx a WHERE a.long_name = t.name
    )
    ORDER BY t.name
    LIMIT 20
    """
)

hierarchical_tag_examples = query_all(
    conn,
    """
    SELECT name, path, description
    FROM tags_list
    WHERE instr(path, '/') > 0
    ORDER BY path
    LIMIT 20
    """
)

print(tags_summary)
print()
print(markdown_table(duplicate_tag_rows, ["name", "duplicate_count", "ids"]))
print()
print(markdown_table(overlap_tag_rows, ["name", "path", "description"]))


### 结果解读

这一段是目前最能直接支持团队决策的新证据。
它回答了两个关键问题：
- Tags 到底是不是天然适合做 hierarchy？
- Tags 和 Activist Codes 之间的重叠是不是已经大到需要治理？


In [ ]:
tag_overlap_rate = round(tags_summary["exact_name_overlap_with_activist_codes"] / tags_summary["total_tags"] * 100, 2)
tag_hierarchy_rate = round(tags_summary["hierarchical_paths"] / tags_summary["total_tags"] * 100, 2)
print(f"Hierarchical tag rate: {tag_hierarchy_rate}%")
print(f"Exact tag/code overlap rate: {tag_overlap_rate}%")
print()
print(
    "Interpretation: Tags are already mostly hierarchical, which supports using them as the roll-up layer. "
    "At the same time, the overlap with Activist Code names is too large to ignore, so a deduplication and role-clarity cleanup is necessary."
)


## 8. 看应用集中度：是不是少数 code 占据了大多数使用量

如果使用非常集中，说明：
- 一部分 code 是核心资产
- 另一大部分 code 可能只是小众或一次性名单标签

这会直接影响治理策略：
我们不应该用同一套管理方式对待“核心 code”和“一次性导入 code”。


In [ ]:
code_usage_rows = query_all(
    conn,
    """
    SELECT activistcodeid, activistcodetype, activistcodename, COUNT(*) AS applied_count
    FROM all_activist_codes_applied
    GROUP BY activistcodeid, activistcodetype, activistcodename
    ORDER BY applied_count DESC
    """
)

counts = [row["applied_count"] for row in code_usage_rows]
top10_share = round(sum(counts[:10]) / sum(counts) * 100, 2)
top20_share = round(sum(counts[:20]) / sum(counts) * 100, 2)
median_usage = sorted(counts)[len(counts) // 2]
codes_le_10 = sum(1 for c in counts if c <= 10)
codes_le_100 = sum(1 for c in counts if c <= 100)

usage_bucket_rows = [
    {"usage_bucket": "<=10", "code_count": codes_le_10},
    {"usage_bucket": "11-100", "code_count": sum(1 for c in counts if 11 <= c <= 100)},
    {"usage_bucket": "101-1000", "code_count": sum(1 for c in counts if 101 <= c <= 1000)},
    {"usage_bucket": ">1000", "code_count": sum(1 for c in counts if c > 1000)},
]

print(f"Top 10 codes share of all applications: {top10_share}%")
print(f"Top 20 codes share of all applications: {top20_share}%")
print(f"Median usage per code: {median_usage}")
print(f"Codes used 10 times or fewer: {codes_le_10}")
print(f"Codes used 100 times or fewer: {codes_le_100}")
print()
print(markdown_table(usage_bucket_rows, ["usage_bucket", "code_count"]))


### 结果解读

如果 top 10 或 top 20 code 占据了非常高比例，
说明应该把“治理重点”放在高频 code 和命名标准上，
而不是一开始就试图对所有低频 code 做复杂建模。


In [ ]:
print(
    "Interpretation: usage is highly concentrated. A relatively small number of codes carries most of the operational load, "
    "while many codes are low-frequency or one-off. This strongly suggests a two-speed governance model: "
    "strict standards for core codes, lighter archival rules for low-frequency legacy codes."
)


## 9. 看现有使用模式是不是支持“上传名单打多个 code”

Linda 邮件里最具体的问题之一是：
“Maricopa County PCs 上传时，是不是应该打两个 Activist Codes？”

这里我们从两个角度看：
- 当前系统里有没有“地理/县/PC”这类 code
- 应用表的 input type 是否说明 bulk upload 是主要使用场景之一


In [ ]:
geography_pc_examples = query_all(
    conn,
    """
    SELECT type, long_name
    FROM activist_codes_list_xlsx
    WHERE long_name LIKE '%County%' OR long_name LIKE '% PC%' OR long_name LIKE '%PC%'
    ORDER BY type, long_name
    LIMIT 20
    """
)

inputtype_rows = query_all(
    conn,
    """
    SELECT inputtype, COUNT(*) AS application_count
    FROM all_activist_codes_applied
    GROUP BY inputtype
    ORDER BY application_count DESC
    """
)

print(markdown_table(geography_pc_examples, ["type", "long_name"]))
print()
print(markdown_table(inputtype_rows, ["inputtype", "application_count"]))


### 结果解读

这一步主要是回答：
- 现有系统是否已经有“角色 / 地理 / 名单来源”混在一起的趋势
- 多个 code 叠加是否符合真实使用场景


In [ ]:
print(
    "Interpretation: the current dictionary already contains geography and PC-related labels, and the application data is dominated by Back End and Bulk workflows. "
    "So yes: multiple activist codes per uploaded list is not only feasible, it is already consistent with how the system has been used."
)


## 10. 把数据证据翻译成解决方案

到这里，我们已经知道：
- Activist Codes 体系是真正在被大量使用的
- 名字基本能对齐，但缺正式字典主键
- short name 冲突明显
- 有少量未使用定义和极少数 unmatched code
- bulk upload 是主要应用场景之一

所以下一步不是再“讨论概念”，而是可以给出一套可以执行的治理和数据科学方案。


In [ ]:
solution_rows = [
    {
        "solution_id": "S1",
        "recommendation": "建立 canonical Activist Code dictionary",
        "what_to_build": "新增一张主字典表，至少包含：code_id、canonical_long_name、type、purpose_class、status、owner、created_date、retired_date。",
        "why_data_supports_it": "当前定义表没有可直接联结的 ID，只能靠名字匹配，后续治理风险高。",
    },
    {
        "solution_id": "S2",
        "recommendation": "把 Activist Codes 设计成原子标签，而不是大杂烩",
        "what_to_build": "对上传名单使用多 code 规则，例如：Role=PC、Geo=MaricopaCounty、Year=2025、Source=Upload。",
        "why_data_supports_it": "Bulk/Back End 应用占绝对多数，且现有字典已存在 geography/PC 类 code，说明多标签模式是自然延伸。",
    },
    {
        "solution_id": "S3",
        "recommendation": "把 Tags 重新定义为 roll-up 层",
        "what_to_build": "例如 `DemLeadership > PrecinctCommittee > CountyPCs` 这样的层级 Tag，用来汇总多个 Activist Codes。",
        "why_data_supports_it": "Tag 数据里 264/332 已经是分层 path，同时 191 个 tag 名称与 Activist Code 完全重叠，说明现在最需要的是角色分工而不是继续混用。",
    },
    {
        "solution_id": "S4",
        "recommendation": "做重复/冲突检测",
        "what_to_build": "建立 short-name 冲突表、未使用定义表、未映射应用 code 表、相似名称候选表。",
        "why_data_supports_it": "short name 冲突已被实证验证，Tags 里也已发现 duplicate name 和大规模 tag/code overlap。",
    },
    {
        "solution_id": "S5",
        "recommendation": "建立 upload-time code recommender",
        "what_to_build": "根据上传文件名、来源、年份、地理区域、角色关键词，自动推荐应打哪些 Activist Codes 与 Tags。",
        "why_data_supports_it": "邮件里的场景高度规则化，例如 `Maricopa County PCs` 这种文件名本身就能拆出角色 + 地理 + 年份。",
    },
    {
        "solution_id": "S6",
        "recommendation": "做 code 生命周期管理",
        "what_to_build": "把 code 分成 Core / Managed / Legacy / Retired 四类，并做年度审计。",
        "why_data_supports_it": "使用集中度很高，很多低频 code 不能和高频核心 code 一样管理。",
    },
]

print(markdown_table(solution_rows, ["solution_id", "recommendation", "what_to_build", "why_data_supports_it"]))


### 结果解读

这里的核心思路不是“删掉所有旧 code”，
而是把体系拆成四层：
- 核心原子 code
- 层级 tags
- 自动推荐
- 定期审计


## 11. 对 Linda 邮件中的具体问题给出直接回答

Linda 最想知道的，不是抽象框架，而是：
“Maricopa County PCs 上传时，到底该怎么打？”


In [ ]:
direct_answer_rows = [
    {
        "question": "上传 Maricopa County PCs 时，是否应该打两个 Activist Codes？",
        "answer": "是，建议至少打两个原子 Activist Codes：一个表示角色/身份，一个表示地理。",
    },
    {
        "question": "推荐的 Activist Codes 组合是什么？",
        "answer": "建议拆成 `Role: Precinct Committee`、`Geo: Maricopa County`，必要时再加 `Year: 2025` 或 `Source: Upload`。",
    },
    {
        "question": "Tag 应该怎么用？",
        "answer": "Tag 不替代 Activist Codes，而是做 roll-up，例如 `DemLeadership > PCs`，把多个相关 Activist Codes 汇总起来。",
    },
    {
        "question": "当前最应该先做什么？",
        "answer": "先冻结新命名规则，再清理 short-name 冲突、7 个未使用 code、1 个未映射应用 code，以及 `EDU` duplicate tag 和 191 个 tag/code 重叠项。",
    },
]

print(markdown_table(direct_answer_rows, ["question", "answer"]))


## 12. 生成图表

这里生成三张图：
- 定义表按 type 分布
- 应用表按 type 分布
- 使用强度分桶


In [ ]:
definition_type_figure = FIGURE_DIR / "activist_codes_definition_by_type.svg"
application_type_figure = FIGURE_DIR / "activist_codes_application_by_type.svg"
usage_bucket_figure = FIGURE_DIR / "activist_codes_usage_buckets.svg"
tag_governance_figure = FIGURE_DIR / "activist_tags_governance_signals.svg"

write_svg_bar_chart(
    definition_type_figure,
    "Activist Code Definitions by Type",
    [(row["type"], row["definition_count"]) for row in definition_type_rows[:10]],
    bar_color="#0f766e",
)

write_svg_bar_chart(
    application_type_figure,
    "Activist Code Applications by Type",
    [(row["activistcodetype"], row["application_count"]) for row in application_type_rows[:10]],
    bar_color="#c2410c",
)

write_svg_bar_chart(
    usage_bucket_figure,
    "Used Codes by Usage Bucket",
    [(row["usage_bucket"], row["code_count"]) for row in usage_bucket_rows],
    bar_color="#4f46e5",
)

write_svg_bar_chart(
    tag_governance_figure,
    "Tag Governance Signals",
    [
        ("hierarchical", tags_summary["hierarchical_paths"]),
        ("flat", tags_summary["path_equals_name"]),
        ("overlap", tags_summary["exact_name_overlap_with_activist_codes"]),
    ],
    bar_color="#9333ea",
)

print(definition_type_figure)
print(application_type_figure)
print(usage_bucket_figure)
print(tag_governance_figure)


## 13. 输出成 markdown 报告

探索结束后，把关键信息固化成报告，方便给团队讨论。


In [ ]:
report_lines = [
    "# Activist Codes Requirements Analysis",
    "",
    "## Requirement Extraction",
    "",
    markdown_table(requirements, ["requirement_id", "business_need", "why_it_matters", "data_question"]),
    "",
    "## Data Availability",
    "",
    markdown_table([data_availability], ["activist_code_definitions", "activist_code_applications", "distinct_applied_code_ids", "distinct_applied_code_names", "tag_definitions", "duplicate_tag_name_groups", "exact_tag_code_name_overlap"]),
    "",
    "Interpretation: both Activist Code and Tag data are now available, so duplicate and overlap cleanup can be evidence-based rather than hypothetical.",
    "",
    "## Definitions by Type",
    "",
    markdown_table(definition_type_rows, ["type", "definition_count"]),
    "",
    "## Applications by Type",
    "",
    markdown_table(application_type_rows, ["activistcodetype", "application_count"]),
    "",
    "Interpretation: Constituency dominates both the code catalog and actual usage, so audience grouping is already the main operational pattern.",
    "",
    "## Short-Name Conflicts",
    "",
    markdown_table(short_conflict_rows, ["short", "duplicate_count", "long_names"]),
    "",
    f"Interpretation: short names have {short_conflict_summary['duplicate_short_groups']} duplicate groups, so they should not be used as canonical keys.",
    "",
    "## Definition / Application Matching",
    "",
    markdown_table([match_summary], ["match_long_name", "match_medium", "match_short", "distinct_applied_codes"]),
    "",
    markdown_table(unmatched_applied_rows, ["activistcodetype", "activistcodename"]),
    "",
    f"Interpretation: long-name matching covers {long_name_match_rate}% of applied codes, which makes dictionary governance feasible, but a formal ID key is still needed.",
    "",
    "## Unused Definitions",
    "",
    markdown_table([unused_definition_summary], ["total_defs", "defs_seen_in_applied", "defs_not_seen_in_applied"]),
    "",
    markdown_table(unused_definition_examples, ["type", "long_name", "medium", "short"]),
    "",
    "## Usage Concentration",
    "",
    markdown_table(usage_bucket_rows, ["usage_bucket", "code_count"]),
    "",
    f"Interpretation: top 10 codes account for {top10_share}% of all applications and top 20 codes account for {top20_share}%. Governance should focus first on high-frequency codes.",
    "",
    "## Tags Audit",
    "",
    markdown_table([tags_summary], ["total_tags", "hierarchical_paths", "path_equals_name", "missing_description", "duplicate_tag_name_groups", "duplicate_tag_extra_rows", "exact_name_overlap_with_activist_codes"]),
    "",
    markdown_table(duplicate_tag_rows, ["name", "duplicate_count", "ids"]),
    "",
    markdown_table(overlap_tag_rows, ["name", "path", "description"]),
    "",
    f"Interpretation: {tag_hierarchy_rate}% of tags are already hierarchical, while {tag_overlap_rate}% have exact name overlap with Activist Codes. This supports using Tags as the hierarchy layer and cleaning the overlapping names.",
    "",
    "## Upload Pattern Evidence",
    "",
    markdown_table(geography_pc_examples, ["type", "long_name"]),
    "",
    markdown_table(inputtype_rows, ["inputtype", "application_count"]),
    "",
    "Interpretation: multiple activist codes per uploaded list is consistent with the current system and workflow patterns.",
    "",
    "## Recommended Data Science / Governance Solutions",
    "",
    markdown_table(solution_rows, ["solution_id", "recommendation", "what_to_build", "why_data_supports_it"]),
    "",
    "## Direct Answer to the Email Thread",
    "",
    markdown_table(direct_answer_rows, ["question", "answer"]),
    "",
    "## Figures",
    "",
    f"![Definitions by type](./figures/{definition_type_figure.name})",
    "",
    f"![Applications by type](./figures/{application_type_figure.name})",
    "",
    f"![Usage buckets](./figures/{usage_bucket_figure.name})",
    "",
    f"![Tag governance](./figures/{tag_governance_figure.name})",
]

REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
print(REPORT_PATH)

conn.close()
